In [10]:
# ==============================
# 1. INSTALL (run once)
# ==============================
# pip install transformers sentence-transformers torch rapidfuzz

# ==============================
# 2. IMPORTS
# ==============================
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
import re



In [11]:
# ==============================
# 3. LOAD MODELS
# ==============================

# Toxicity model
toxicity_model = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    return_all_scores=True
)

# Moderation model
moderation_model = pipeline(
    "text-classification",
    model="KoalaAI/Text-Moderation",
    return_all_scores=True
)

# Semantic model (MiniLM)
embedder = SentenceTransformer('all-MiniLM-L6-v2')



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24031.67it/s]
RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 5766.75it/s]
DebertaForSequenceClassification LOAD REPORT from: KoalaAI/Text-Moderation
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6733.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                 

In [12]:
# ==============================
# 4. NORMALIZATION
# ==============================

def normalize_text(text):
    text = text.lower()
    
    # Replace leetspeak
    replacements = {
        "0": "o", "1": "i", "3": "e",
        "@": "a", "$": "s"
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    
    # Remove extra symbols
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    return text



In [13]:
# ==============================
# 5. FUZZY MATCHING
# ==============================

SUSPICIOUS_PHRASES = [
    "meet me",
    "dont tell anyone",
    "keep this secret",
    "come alone",
]

def fuzzy_score(text):
    score = 0
    for phrase in SUSPICIOUS_PHRASES:
        score = max(score, fuzz.partial_ratio(text, phrase) / 100)
    return score



In [14]:
# ==============================
# 6. RULE-BASED GROOMING
# ==============================

def rule_grooming_score(messages):
    score = 0
    
    for msg in messages:
        msg = msg.lower()

        if any(x in msg for x in ["meet", "come over", "visit"]):
            score += 1.5

        if any(x in msg for x in ["secret", "dont tell", "keep this"]):
            score += 2.5

        if any(x in msg for x in ["alone", "private"]):
            score += 2

        if any(x in msg for x in ["special", "trust me"]):
            score += 1.5

    return min(score / 5, 1)



In [15]:
# ==============================
# 7. SEMANTIC GROOMING (MiniLM)
# ==============================

GROOMING_PATTERNS = [
    "let's keep this between us",
    "don't tell your parents",
    "you are special to me",
    "we should meet privately",
    "i understand you better than others",
    "this is our secret"
]

pattern_embeddings = embedder.encode(GROOMING_PATTERNS)

def semantic_score(messages):
    text = " ".join(messages[-5:])
    emb = embedder.encode(text)
    scores = util.cos_sim(emb, pattern_embeddings)
    return float(scores.max())



In [16]:
# ==============================
# 8. MODEL HELPERS
# ==============================

def get_toxicity_score(text):
    results = toxicity_model(text)

    # Handle both cases safely
    if isinstance(results, list):
        results = results[0]

    if isinstance(results, dict):
        return results["score"]

    # If list of dicts
    return max(r['score'] for r in results)

def get_moderation_scores(text):
    results = moderation_model(text)

    if isinstance(results, list):
        results = results[0]

    # If single dict
    if isinstance(results, dict):
        return {
            "sexual": results.get("score", 0),
            "violence": 0,
            "hate": 0
        }

    # If list of dicts
    scores = {r['label']: r['score'] for r in results}

    return {
        "sexual": scores.get("sexual", 0),
        "violence": scores.get("violence", 0),
        "hate": scores.get("hate", 0)
    }



In [17]:
# ==============================
# 9. FINAL PIPELINE
# ==============================

def detect_grooming(messages):
    text = normalize_text(" ".join(messages))

    # ML signals
    tox_score = get_toxicity_score(text)
    mod_scores = get_moderation_scores(text)

    # Rule + fuzzy
    rule_score = rule_grooming_score(messages)
    fuzzy = fuzzy_score(text)

    # Semantic
    sem_score = semantic_score(messages)

    # Final weighted score
    final_score = (
        0.2 * tox_score +
        0.2 * mod_scores["sexual"] +
        0.2 * rule_score +
        0.2 * sem_score +
        0.2 * fuzzy
    )

    # Risk level
    if final_score > 0.7:
        level = "HIGH"
    elif final_score > 0.4:
        level = "MEDIUM"
    else:
        level = "LOW"

    return {
        "risk_score": round(final_score, 3),
        "level": level,
        "signals": {
            "toxicity": tox_score,
            "sexual": mod_scores["sexual"],
            "rule": rule_score,
            "semantic": sem_score,
            "fuzzy": fuzzy
        }
    }



In [27]:
# ==============================
# 10. TEST
# ==============================

messages = [[
    "hey you're really mature for your age",
    "i think we understand each other",
    "dont tell anyone about this",
    "we should meet privately sometime"
], [
    "hey you're really mature for your age",
    "i think we should make 3 pipeline for training testing and production",
    "This is great work, lets focus on the next task now",
    "we should meet and see if we can get this issue resolved"
],  [
    "hey how was your day?",
    "did you finish your homework?",
    "let's play a game later",
    "see you tomorrow!"
],  [
    "don’t tell your parents about this",
    "this is just between us",
    "they wouldn’t understand",
    "keep this secret okay?"
],  [
    "d0nt t3ll any0ne",
    "this is our secr3t",
    "lets m33t al0ne",
    "just us"
],  [
    "this is a secret recipe 😂",
    "don’t tell anyone my surprise plan",
    "we are meeting at the cafe",
    "bring your friends too"
]
]

for i in messages:
    result = detect_grooming(i)
    print(i, "result = ",  result)


["hey you're really mature for your age", 'i think we understand each other', 'dont tell anyone about this', 'we should meet privately sometime'] result =  {'risk_score': 0.716, 'level': 'HIGH', 'signals': {'toxicity': 0.0027796293143182993, 'sexual': 0.9780576825141907, 'rule': 1, 'semantic': 0.5982235670089722, 'fuzzy': 1.0}}
["hey you're really mature for your age", 'i think we should make 3 pipeline for training testing and production', 'This is great work, lets focus on the next task now', 'we should meet and see if we can get this issue resolved'] result =  {'risk_score': 0.435, 'level': 'MEDIUM', 'signals': {'toxicity': 0.0006711010355502367, 'sexual': 0.9921505451202393, 'rule': 0.3, 'semantic': 0.16932988166809082, 'fuzzy': 0.7142857142857143}}
['hey how was your day?', 'did you finish your homework?', "let's play a game later", 'see you tomorrow!'] result =  {'risk_score': 0.344, 'level': 'LOW', 'signals': {'toxicity': 0.001558199874125421, 'sexual': 0.8857443332672119, 'rule